# 03 — Iterative Debugging Loop & Execution Reward Design
Demonstrate error injection, sandbox execution feedback extraction, reward function outputs, and multi-turn agentic self-correction.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.error_injection import BugInjector, BugType
from src.execution.executor import PythonSandbox
from src.rewards.execution_reward import compute_binary_reward, compute_partial_reward, compute_status_aware_reward
from src.debugging.debug_loop import DebugLoop

injector = BugInjector(seed=42)
correct_code = "def multiply(a, b):\n    return a * b"

print("=== Synthetic Bug Injection ===")
for category in list(BugType):
    res = injector.inject_bug(correct_code, category=category)
    print(f"[{res['bug_type'].upper()}] -> {res['description']}")

sandbox = PythonSandbox()
test_cases = [{"fn_name": "multiply", "input": [3, 4], "expected": 12}]

def mock_agent_model(prompt: str) -> str:
    if "previous solution failed" in prompt.lower():
        return "```python\ndef multiply(a, b):\n    return a * b\n```"
    return "```python\ndef multiply(a, b):\n    return a + b\n```"

print("\n=== Multi-Turn Agentic Debug Session ===")
loop = DebugLoop(sandbox=sandbox, max_turns=3)
problem_desc = "Write a Python function multiply(a, b) returning product of a and b."
session = loop.run_session(problem_desc, test_cases, model_fn=mock_agent_model)

print(f"Solved: {session['solved']} at Turn {session['solved_turn']}")
for turn in session["history"]:
    print(f"Turn {turn['turn']}: Status={turn['status']} | Reward={turn['reward']:.2f}")
